# Notebook 8 - Automated Commentary

this generates a short research note style paragraph for each quarter describing what happened - biggest moves, portfolio value change, turnover trend etc.

everything is generated from the actual data so if you switch to a different fund manager the commentary automatically describes that manager instead.

this was actually one of my favourite parts to build because turning numbers into sentences is harder than it sounds

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
from src import commentary, config
from src.utils import load_parquet

holdings = load_parquet(config.HOLDINGS_PARQUET)
tx       = load_parquet(config.TRANSACTIONS_PARQUET)
summary  = load_parquet(config.PORTFOLIO_PARQUET)

# commentary for the most recent quarter
latest = summary["quarter"].iloc[-1]
print(f"--- {latest} ---")
print(commentary.quarterly_commentary(latest, holdings, tx, summary))

In [ ]:
# full report covering all quarters
report = commentary.full_report(holdings, tx, summary)

# save to file
from pathlib import Path
out = Path("reports/portfolio_review.md")
out.parent.mkdir(exist_ok=True)
out.write_text(report, encoding="utf-8")
print("report saved to", out)
print()

# show in notebook
from IPython.display import Markdown
Markdown(report)

---
**note on what this data covers**

13F filings only show long positions in US listed securities. they dont show short positions, cash, bonds in some cases, or international holdings. also the data is delayed by 45 days so its not real time. so this is a useful picture of where a manager is allocating capital but its not the complete picture of what they are doing.